In [1]:
# import the functions from collector.py
import sys
import os
sys.path.append(os.path.abspath(".."))

from bjarne_api.collector_new import get_station_details, get_journeys, get_weather, Fetcher
import pandas as pd

In [3]:

# Ausführung im Hauptprogramm 
if __name__ == "__main__":
    fetcher = Fetcher()
    station_name = "Berlin Hbf" 
    
    print(f"Suche Station ID für {station_name}...")
    station_id = fetcher.get_station_id(station_name)
    
    if station_id:
        print(f"Station ID gefunden: {station_id}. Suche Fernverkehrsabfahrten (nächste 60min)...")
        departures = fetcher.stations_details(station_id)
        
        if departures:
            print(f"{len(departures)} Abfahrten gefunden. Lade Details...")
            
            all_trips_dfs = []
            
            
            for i, departure in enumerate(departures, start=1):
                if "tripId" in departure:
                    trip_id = departure["tripId"]
                    line_name = departure.get("line", {}).get("name", "Unbekannt")
                    
                    print(f"Lade Trip {i}/{len(departures)}: {line_name}")
                    
                    trip_data = fetcher.trip_details(trip_id)
                    if trip_data is not None:
                        df_trip = fetcher.create_dataframe(trip_data, ride_id=i)
                    
                    if not df_trip.empty:
                        all_trips_dfs.append(df_trip)
                        
                    else: 
                        print(f"Details für Trip {line_name} konnten nicht verarbeitete werden")
            
            # Zusammenfügen aller DataFrames
            if all_trips_dfs:
                final_df = pd.concat(all_trips_dfs, ignore_index=True)
                print("\n--- FERTIG ---")
                print(final_df)
                print(f"\nGesamtanzahl Haltepunkte analysiert: {len(final_df)}")
                print(f"Anzahl Züge: {final_df['ride_id'].nunique()}")
            else:
                print("Konnte keine Trip-Details verarbeiten.")
        else:
            print("Keine passenden Abfahrten im Zeitraum gefunden.")
    else:
        print("Station nicht gefunden.")

Suche Station ID für Berlin Hbf...
Station ID gefunden: 8011160. Suche Fernverkehrsabfahrten (nächste 60min)...
16 Abfahrten gefunden. Lade Details...
Lade Trip 1/16: ICE 697
Lade Trip 2/16: ICE 2847
Lade Trip 3/16: ICE 1605
Lade Trip 4/16: IC 2172
Lade Trip 5/16: ICE 504
Lade Trip 6/16: ICE 842
Lade Trip 7/16: ICE 852
Lade Trip 8/16: ICE 805
Lade Trip 9/16: FLX 1247
Lade Trip 10/16: BUS 40431
Lade Trip 11/16: EN 40457
Lade Trip 12/16: NJ 457
Lade Trip 13/16: IC 2275
Lade Trip 14/16: ICE 607
Lade Trip 15/16: ICE 679
Lade Trip 16/16: ICE 2943

--- FERTIG ---
     ride_id train_name train_type         station_start         station_dest  \
0          1    ICE 697        ICE  Berlin Gesundbrunnen   Frankfurt(Main)Hbf   
1          1    ICE 697        ICE  Berlin Gesundbrunnen   Frankfurt(Main)Hbf   
2          1    ICE 697        ICE  Berlin Gesundbrunnen   Frankfurt(Main)Hbf   
3          1    ICE 697        ICE  Berlin Gesundbrunnen   Frankfurt(Main)Hbf   
4          1    ICE 697        

In [ ]:
### FILTER FUNCTION FOR ICE/IC TRAINS ONLY ###
def filter_train_type(df):
    output_df = df[df["train_type"].str.contains("ICE|IC", case=False, na=False)].copy()
    return output_df

filtered_df = filter_train_type(final_df)

In [ ]:
### FUNCTION TO FIND POSSIBLE DESTINATIONS FROM A GIVEN STATION ###
def possible_destinations(final_df, station_name):
    possible_destinations = []

    for ride_id, train_group in final_df.groupby("ride_id"):
        
        # find row where current_station is station_name
        start_row = train_group[train_group["current_station"] == station_name]
        
        # If the train actually stops at our station:
        if not start_row.empty:
            station_time = start_row["actual_departure"].iloc[0]
            
            # get stations where actual_arrival is greater than station_time
            later_stops = train_group[train_group["actual_arrival"] > station_time]
            
            # get station names and add to possible_destinations
            stations_names = later_stops["current_station"].tolist()
            possible_destinations.extend(stations_names)
            
    # use set() to remove duplicates
    return set(possible_destinations)

# Example usage:   

possible_destinations(filtered_df, "Berlin Hbf")

{'Bamberg',
 'Berlin Gesundbrunnen',
 'Berlin Südkreuz',
 'Berlin Zoologischer Garten',
 'Berlin-Spandau',
 'Bitterfeld',
 'Bochum Hbf',
 'Braunschweig Hbf',
 'Dortmund Hbf',
 'Dresden Hbf',
 'Dresden-Neustadt',
 'Eisenach Hbf',
 'Elsterwerda',
 'Erfurt Hbf',
 'Erlangen',
 'Essen Hbf',
 'Flughafen BER',
 'Frankfurt(Main)Hbf',
 'Fulda',
 'Göttingen',
 'Hagen Hbf',
 'Halle(Saale)Hbf',
 'Hamburg Hbf',
 'Hamburg-Altona',
 'Hamm(Westf)Hbf',
 'Hannover Hbf',
 'Hildesheim Hbf',
 'Kassel-Wilhelmshöhe',
 'Köln Hbf',
 'Leipzig Hbf',
 'Lutherstadt Wittenberg Hbf',
 'München Hbf',
 'Neustrelitz Hbf',
 'Nürnberg Hbf',
 'Oranienburg',
 'Rostock Hbf',
 'Salzwedel',
 'Stendal Hbf',
 'Uelzen',
 'Waren(Müritz)',
 'Wolfsburg Hbf',
 'Wuppertal Hbf'}

In [ ]:
import pandas as pd

### FUNCTION TO GET CONNECTION BETWEEN TWO STATIONS ###
def get_connection(df, station_start, station_dest):
    
    # check if station_dest is reachable from station_start
    valid_destinations = possible_destinations(df, station_start)
    
    if station_dest not in valid_destinations:
        print(f"{station_dest} is not reachable from {station_start}.")
        print(f"Possible options are: {sorted(list(valid_destinations))}")
        return None

    
    suited_rides = []

    for ride_id, train_group in df.groupby("ride_id"):
        start_row = train_group[train_group["current_station"] == station_start]
        
        if not start_row.empty:
            station_start_time = start_row["actual_departure"].iloc[0]
            
            # get stations coming after station_start_time
            later_stops = train_group[train_group["actual_arrival"] > station_start_time]
            
            # check: is station_dest in later_stops?
            if station_dest in later_stops["current_station"].values:
                # add ride_id to suited_rides
                suited_rides.append(train_group)

    # combine all suited_rides into one df
    if suited_rides:
        final_connections = pd.concat(suited_rides, ignore_index=True)
        num_trains = final_connections['ride_id'].nunique()
        print(f"Found {len(suited_rides)} different trains connecting {station_start} to {station_dest}.")
        return final_connections
    else:
        print(f"No current connections found.")
        return None

# example usage:
df_connections = get_connection(filtered_df, "Berlin Hbf", "Braunschweig Hbf")

Found 2 different trains connecting Berlin Hbf to Braunschweig Hbf.


In [20]:
### FUNCTION TO GET DF WITH RIDES BETWEEN TWO STATIONS ###
def get_connections_between_stations(df, start_station, end_station):

    df_temp = filter_train_type(df)
    
    return get_connection(df_temp, start_station, end_station)

get_connections_between_stations(final_df, "Berlin Hbf", "Braunschweig Hbf")

Found 2 different trains connecting Berlin Hbf to Braunschweig Hbf.


,ride_id,train_name,train_type,station_start,station_dest,current_station,planned_arrival,actual_arrival,planned_departure,actual_departure,current_delay,start_temp,start_precip,start_wind_speed,end_temp,end_precip,end_wind_speed
0,15,ICE 679,ICE,Berlin Ostbahnhof,Kassel-Wilhelmshöhe,Berlin Ostbahnhof,None,None,2026-02-05T20:14:00+01:00,2026-02-05T20:14:00+01:00,0,-1.4,0.1,10.1,-0.1,0.0,4.3
1,15,ICE 679,ICE,Berlin Ostbahnhof,Kassel-Wilhelmshöhe,Berlin Hbf,2026-02-05T20:25:00+01:00,2026-02-05T20:25:00+01:00,2026-02-05T20:29:00+01:00,2026-02-05T20:29:00+01:00,0,-1.4,0.1,10.1,-0.1,0.0,4.3
2,15,ICE 679,ICE,Berlin Ostbahnhof,Kassel-Wilhelmshöhe,Berlin Zoologischer Garten,2026-02-05T20:33:00+01:00,2026-02-05T20:33:00+01:00,2026-02-05T20:35:00+01:00,2026-02-05T20:35:00+01:00,0,-1.4,0.1,10.1,-0.1,0.0,4.3
3,15,ICE 679,ICE,Berlin Ostbahnhof,Kassel-Wilhelmshöhe,Berlin-Spandau,2026-02-05T20:43:00+01:00,2026-02-05T20:43:00+01:00,2026-02-05T20:45:00+01:00,2026-02-05T20:45:00+01:00,0,-1.4,0.1,10.1,-0.1,0.0,4.3
4,15,ICE 679,ICE,Berlin Ostbahnhof,Kassel-Wilhelmshöhe,Wolfsburg Hbf,2026-02-05T21:37:00+01:00,2026-02-05T21:37:00+01:00,2026-02-05T21:39:00+01:00,2026-02-05T21:39:00+01:00,0,-1.4,0.1,10.1,-0.1,0.0,4.3
5,15,ICE 679,ICE,Berlin Ostbahnhof,Kassel-Wilhelmshöhe,Braunschweig Hbf,2026-02-05T21:55:00+01:00,2026-02-05T21:55:00+01:00,2026-02-05T21:57:00+01:00,2026-02-05T21:57:00+01:00,0,-1.4,0.1,10.1,-0.1,0.0,4.3
6,15,ICE 679,ICE,Berlin Ostbahnhof,Kassel-Wilhelmshöhe,Hildesheim Hbf,2026-02-05T22:18:00+01:00,2026-02-05T22:18:00+01:00,2026-02-05T22:19:00+01:00,2026-02-05T22:19:00+01:00,0,-1.4,0.1,10.1,-0.1,0.0,4.3
7,15,ICE 679,ICE,Berlin Ostbahnhof,Kassel-Wilhelmshöhe,Göttingen,2026-02-05T22:50:00+01:00,2026-02-05T22:50:00+01:00,2026-02-05T22:52:00+01:00,2026-02-05T22:52:00+01:00,0,-1.4,0.1,10.1,-0.1,0.0,4.3
8,15,ICE 679,ICE,Berlin Ostbahnhof,Kassel-Wilhelmshöhe,Kassel-Wilhelmshöhe,2026-02-05T23:13:00+01:00,2026-02-05T23:13:00+01:00,None,None,0,-1.4,0.1,10.1,-0.1,0.0,4.3
9,16,ICE 2943,ICE,Berlin Ostbahnhof,Kassel-Wilhelmshöhe,Berlin Ostbahnhof,None,None,2026-02-05T20:14:00+01:00,2026-02-05T20:14:00+01:00,0,-1.4,0.1,10.1,-0.1,0.0,4.3
